In [1]:
import os
import pandas as pd
import toml
from openpyxl.styles import Font, Border, Side, Alignment

# This is only needed for years in which we summarize VMT and not emissions 
# e.g., RTP 2035 and 2050 results: Y:\Air Quality\King County Emissions Inventory\RTP_2026_2050

# Specify config 
config_path = "../configs/config_vmt_forecast.toml"
config = toml.load(config_path)

# Excel formatting
no_border = Border(
    left=Side(border_style=None),
    right=Side(border_style=None),
    top=Side(border_style=None),
    bottom=Side(border_style=None)
)

In [63]:
for county in ["King", "Kitsap", "Pierce", "Snohomish"]:
    print(county)
    df_results = pd.DataFrame()

    # Get list of CSVs from directory
    city_list = [f.split(".csv")[0] for f in os.listdir(os.path.join(config['output_root'],"data", "city", county,"interzonal_vmt")) if f.endswith('.csv')]

    for city in city_list:
        # Load data
        df_interzonal_vmt = pd.read_csv(os.path.join(config['output_root'],"data", "city", county,"interzonal_vmt", f"{city}.csv"))
        df_intrazonal_vmt = pd.read_csv(os.path.join(config['output_root'],"data", "city", county,"intrazonal_vmt", f"{city}.csv"))
        df_intra = df_intrazonal_vmt.groupby('vehicle_type').sum()[["VMT"]]
        df_inter = df_interzonal_vmt.sum()

        df_city_results = pd.DataFrame()

        # summarize by passenger vehicles, medium trucks, and heavy truck totals
        for col in ["Passenger Vehicles", "Medium Trucks", "Heavy Trucks"]:
            df_city_results.loc[city,col] = 0 
        for mode in ["sov", "hov2", "hov3"]:
            df_city_results.loc[city,"Passenger Vehicles"] = df_city_results.loc[city,"Passenger Vehicles"] + df_intra.loc[mode, "VMT"] + df_inter.loc[f"{mode}_vmt"]
        for mode in ['medium', 'heavy']:
            df_city_results.loc[city,f"{mode.capitalize()} Trucks"] = df_city_results.loc[city,f"{mode.capitalize()} Trucks"] + df_intra.loc[f"{mode}truck", "VMT"] + df_inter.loc[f"{mode}_truck_vmt"]

        df_city_results = df_city_results.reset_index()
        df_city_results.rename(columns={'index': 'City'}, inplace=True)
        df_results = pd.concat([df_results, df_city_results])

    output_path = os.path.join(config["output_root"], f"city_vmt_summary_{county}.xlsx")
    with pd.ExcelWriter(output_path) as writer:
        df_results.to_excel(writer, index=False, sheet_name=f"VMT Total {config['analysis_year_list'][0]}")
        ws = writer.sheets[f"VMT Total {config['analysis_year_list'][0]}"]

        for row in ws.iter_rows():
            for cell in row:
                cell.border = no_border
                cell.font = Font(bold=False)
                if cell.column == 1:
                    cell.alignment = Alignment(horizontal='center')
                if cell.column > 1 and isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0'

        for col in ws.columns:
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in col)
            ws.column_dimensions[col[0].column_letter].width = max(max_len + 2, 18)

    df_results.style.format({col: '{:,.0f}' for col in df_results.columns if col != 'City'})


King
Kitsap
Pierce
Snohomish
